# CoNR reasoning-trace generation v2 -- independent-solve + self-reflection (teacher: Qwen3-Next-80B-A3B-Thinking)

Replaces the backward-rationalization pipeline in `qwen3-next-80b-conr-trace-gen.ipynb`.
That pipeline told the teacher the gold program/answer up front and asked it to
write a trace that arrives at it. Reading through `train_with_reasoning_trace.json`
and `valid_with_reasoning_trace.json` end-to-end turned up ~1% of traces
(26/2905 train, 6/571 valid) where the teacher's own reasoning contradicted the
question but it followed the given program anyway, sometimes explicitly
acknowledging the mismatch mid-trace before still emitting the "verified"
program -- i.e. real leakage of the backward-rationalization setup, not just a
denylist near-miss.

This version instead:
1. **Independent-solve** -- the teacher only sees pre_text/table/post_text/
   question, exactly like the student model at inference time. No gold program
   or answer is ever in the prompt. This is the same shape teammate Chi's
   distillation pipeline uses (`train_mixed_reasoning_v5.json`), whose
   convention rules (see below) are folded into the prompt here.
2. **Self-reflection** -- before committing to a final `<program>`, the model is
   instructed to open a `<reflection>` block that re-checks its own draft
   against the question and the convention rules, and only then emit the final
   `<program>`. This is the one addition beyond what Chi's pipeline does: Chi
   filters *after* generation (drop non-exact-match), this prompt asks the
   model to self-correct *before* committing.
3. **Post-hoc exact-match filtering** -- same as Chi's approach: only samples
   whose final program exactly matches gold (structurally, via
   `scorer.equal_program`) are kept for SFT. Independent-solve means most
   samples won't match on the first attempt; this is expected and is the
   whole point (it selects for traces that are genuinely reconstructible from
   the context, not ones grounded by being told the answer).

Convention rules baked into the prompt (from Chi's error analysis on the first
independent-solve pass, where raw exact-match was only 14.4% before these were
added, rising to 62.8%/63.4% train/valid):
  - No nested calls in an argument position (e.g. `divide(add(#0,#1), #2)` is
    invalid) -- almost all gold programs are flat chains referencing prior
    steps via `#N`.
  - Never append an extra `multiply(#N, 100)` step to rescale a ratio into a
    percentage unless the question explicitly asks for a percentage figure
    and the gold convention does so too -- this was the single largest error
    category (27.3%).
  - Never wrap a single already-known value in `table_sum(row, none)` just to
    "look up" it -- table_* ops are for aggregating/selecting from a row's
    multiple columns, not restating one cell.
  - Only use the 10 valid operators: add, subtract, multiply, divide, exp,
    greater, table_sum, table_average, table_max, table_min.
  - Always produce a program -- never answer in prose only.
  - `table_*` calls always take `(row_label, none)`, where `row_label` is the
    exact row label from the table's first column -- never a list of numbers.

Run on the same server as before (2x RTX PRO 6000 96GB). Upload `train.json` /
`valid.json` via the Server Web UI file browser before running (adjust
`DATA_JSON_PATH` below).

In [ ]:
%%capture
!pip install -q vllm tabulate
!pip install -q -U "typing_extensions>=4.12" "pydantic>=2.9"
# IMPORTANT: restart the kernel after this cell finishes (Kernel > Restart),
# then re-run from the top -- same stale-import issue as the original notebook.


In [ ]:
import json
from pathlib import Path

# Adjust if you uploaded train.json/valid.json somewhere else via the Server Web UI file browser.
DATA_JSON_PATH = Path("/root/data/train.json")
OUTPUT_DIR = Path("/root/outputs/conr_trace_v2_independent_solve")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_JSON_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_JSON_PATH} not found. Upload train.json/valid.json via the Server Web UI "
        "file browser first, or update DATA_JSON_PATH to match where you put it."
    )

raw_data = json.load(open(DATA_JSON_PATH, encoding="utf-8"))
print(f"Loaded {len(raw_data)} samples from {DATA_JSON_PATH}")


## Context formatting (same as the 0-shot/1-shot/few-shot/SFT notebooks)

In [ ]:
from tabulate import tabulate

def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

samples = []
for i, s in enumerate(raw_data):
    samples.append({
        "train_index": i,
        "pre_text": formatting_pre_text(s),
        "table": formatting_table(s),
        "table_raw": s["table"],
        "post_text": formatting_post_text(s),
        "question": s["qa"]["question"],
        "gold_program": s["qa"]["program"],
        "gold_answer": s["qa"]["exe_ans"],
    })

print(f"Prepared {len(samples)} samples for trace generation.")
samples[0]


## CoNR prompt v2 -- independent-solve + self-reflection

Unlike `CONR_SYSTEM_PROMPT` in the v1 notebook, this prompt never reveals the
gold program or answer. The teacher gets exactly the context/question the
student sees, is told the 10 valid operators and the convention rules learned
from Chi's error analysis, and is asked to draft a program, **reflect on it
against the rules and the question**, then emit a (possibly revised) final
program. Post-generation, only exact matches against gold are kept -- the
reflection step is meant to raise the exact-match rate versus Chi's
generate-then-filter-only approach, not to replace filtering.

In [ ]:
CONR_SYSTEM_PROMPT = """You are a senior financial analyst answering a numerical question about a
Vietnamese financial document. You are given the context (text before a
table, the table itself, text after the table) and the question. You do NOT
know the correct answer in advance -- work it out yourself, the way a real
analyst would.

Your program must be built ONLY from these 10 operators:
add, subtract, multiply, divide, exp, greater,
table_sum, table_average, table_max, table_min

### PROGRAM FORMAT RULES (violating these is the most common failure mode):
1. Every step is a flat call `op(arg1, arg2)` -- NEVER nest a call inside an
   argument (e.g. `divide(add(#0, #1), #2)` is INVALID). If you need a prior
   step's result, reference it as `#N` (0-indexed) in a LATER step instead.
2. Do not invent a rescaling step. Only add `multiply(#N, 100)` to convert a
   ratio into a percentage if the question explicitly asks for a percentage
   and no earlier step already produced one in percentage terms. When in
   doubt, match the units the question asks for -- do not add an extra `*100`
   "just in case".
3. Never wrap a single value you already have in `table_sum(row, none)` (or
   any other table_* op) just to restate it -- table_* ops are for reading a
   row of the table, not for repackaging a number you already computed.
4. `table_sum` / `table_average` / `table_max` / `table_min` always take
   exactly two arguments: `(row_label, none)`, where `row_label` is the
   EXACT row label as it appears verbatim in the table's first column (or
   the header, for a column lookup). Never pass a list of numbers as the
   argument.
5. You must always produce a program. Never answer in prose only, and never
   leave the <program> block empty.

### OUTPUT FORMAT (strict):
<think>
[Your reasoning in Vietnamese, 3-6 sentences. Identify which specific values
are needed and WHERE they come from (quote the exact row/column label from
the table, or the exact sentence from pre_text/post_text), explain WHY each
operator was chosen, and walk through multi-step programs in order, referring
to intermediate results the way the program does (step 1 result, step 2
result, ...).]
</think>
<draft_program>YOUR_FIRST_DRAFT_PROGRAM</draft_program>
<reflection>
[2-4 sentences, in Vietnamese. Re-check your draft against the 5 PROGRAM
FORMAT RULES above and against the question: Does every step use only the
allowed operators? Are there any nested calls to flatten? Did you add an
unjustified *100 step? Does every table_* call use a verbatim row label
with `none`, not a list of numbers? Does the final result actually answer
what the question asks (right units, right direction/sign, right
period/column)? If you find a problem, say what it is and how you are
fixing it. If the draft already satisfies all of this, say so briefly.]
</reflection>
<program>YOUR_FINAL_PROGRAM_AFTER_REFLECTION</program>"""

CONR_USER_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### YOUR TASK:
Write the <think>, <draft_program>, <reflection>, and final <program> blocks
as instructed. Work out the answer yourself -- do not assume any particular
number is correct in advance."""


## Load the teacher model with vLLM

Same setup as the v1 notebook -- `tensor_parallel_size=2`, `moe_backend="triton"`
and `VLLM_USE_FLASHINFER_SAMPLER=0` to work around the two RTX PRO 6000
(SM120) FlashInfer JIT-compile failures, `max_num_seqs=800` to stay under the
Mamba-cache-block ceiling for Qwen3-Next's hybrid attention.

In [ ]:
import os

os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen3-Next-80B-A3B-Thinking"

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=2,
    dtype="bfloat16",
    max_model_len=16384,   # headroom for <think> + <draft_program> + <reflection> + <program>
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
    moe_backend="triton",
    max_num_seqs=800,
)

tokenizer = llm.get_tokenizer()
print(f"Loaded {MODEL_NAME} across 2 GPUs.")


## Generate traces (batched via vLLM)

One `llm.generate()` call over the whole dataset, same continuous-batching
approach as v1. The self-reflection block adds length versus v1's
think+program-only output, so `max_tokens` is raised accordingly.

In [ ]:
prompts = []
for s in samples:
    user_msg = CONR_USER_FRAME.format(
        pre_text=s["pre_text"], table=s["table"], post_text=s["post_text"],
        question=s["question"],
    )
    messages = [
        {"role": "system", "content": CONR_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)

# 6144 (v1 used 4096 for think+program only): the added <draft_program> and
# <reflection> blocks need extra room on top of what v1 budgeted.
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.9,
    max_tokens=6144,
)

outputs = llm.generate(prompts, sampling_params)
print(f"Generated {len(outputs)} responses.")


## Parse + validate each trace

Reuses `notebooks/evaluate/scorer.py` (the one shared PA/EA evaluator in this
repo) for the exact-match check, via `equal_program` -- the same symbolic
equivalence check used to score the student models, rather than a bespoke
structural-match reimplementation. Independent-solve means most drafts will
NOT match gold on the first pass; only exact matches are kept for SFT, exactly
as in Chi's pipeline.

In [ ]:
import sys
sys.path.append(str(Path("/root/repo/notebooks/evaluate")))  # adjust to wherever scorer.py lives on this server
from scorer import program_tokenization, equal_program, extract_program

import re


def extract_blocks(raw_text: str):
    """Pull <think>, <draft_program>, <reflection>, <program> out of raw teacher output.

    Qwen3-Next-Thinking doesn't emit a literal opening `<think>` tag (same
    behavior as v1) -- the chat template already puts the model in "thinking"
    mode at the start of the assistant turn. Treat everything before the
    first `</think>` as the think block regardless of an opening tag.
    """
    close_think_idx = raw_text.find("</think>")
    if close_think_idx == -1:
        think = None
        rest = raw_text
    else:
        think = raw_text[:close_think_idx]
        think = re.sub(r"^\s*<think>\s*", "", think).strip()
        rest = raw_text[close_think_idx + len("</think>"):]

    def _tag(name, text):
        m = re.search(rf"<{name}>(.*?)</{name}>", text, re.DOTALL)
        return m.group(1).strip() if m else None

    draft_program = _tag("draft_program", rest)
    reflection = _tag("reflection", rest)
    program = _tag("program", rest)

    return think, draft_program, reflection, program


LEAK_PATTERNS = [
    r"chương trình\s+được cho",
    r"kết quả\s+được cho",
    r"đáp án\s+được cho",
    r"chương trình\s+được cung cấp",
    r"kết quả\s+được cung cấp",
    r"đáp án\s+được cung cấp",
    r"tuân (theo|thủ) chương trình",
    r"đã biết trước",
    r"đáp án được xác nhận",
    r"\bverified\b",
    r"\bas given\b",
    r"\bwe are told\b",
    r"\bgiven answer\b",
    r"\bgiven program\b",
]
# Independent-solve means the teacher is never told the answer, so these
# leak patterns should not fire in practice -- kept as a defense-in-depth
# check (e.g. if the model hallucinates having been given an answer), not
# because the prompt can leak a gold value that was never in it.


def find_leak_phrases(text: str):
    if not text:
        return []
    lowered = text.lower()
    return [pat for pat in LEAK_PATTERNS if re.search(pat, lowered)]


In [ ]:
results = []
n_exact_match = 0
n_missing_program = 0
n_leak = 0
n_reflection_changed = 0

for s, output in zip(samples, outputs):
    raw_text = output.outputs[0].text
    think, draft_program, reflection, program = extract_blocks(raw_text)

    final_program = program or draft_program  # fall back to draft if final <program> tag missing
    exact_match = False
    if final_program:
        try:
            exact_match = equal_program(
                program_tokenization(s["gold_program"]),
                program_tokenization(extract_program(final_program)),
            )
        except Exception:
            exact_match = False

    leaks = find_leak_phrases(think) + find_leak_phrases(reflection)
    reflection_changed = bool(
        draft_program and program and extract_program(draft_program) != extract_program(program)
    )

    if final_program is None:
        n_missing_program += 1
    if exact_match:
        n_exact_match += 1
    if leaks:
        n_leak += 1
    if reflection_changed:
        n_reflection_changed += 1

    results.append({
        "train_index": s["train_index"],
        "question": s["question"],
        "gold_program": s["gold_program"],
        "gold_answer": s["gold_answer"],
        "raw_output": raw_text,
        "parsed_think": think,
        "parsed_draft_program": draft_program,
        "parsed_reflection": reflection,
        "parsed_program": final_program,
        "exact_program_match": exact_match,
        "reflection_changed_program": reflection_changed,
        "leak_phrases_found": leaks,
    })

print(f"Total: {len(results)}")
print(f"Exact program match: {n_exact_match} ({100 * n_exact_match / len(results):.1f}%)")
print(f"Missing final <program> tag (draft/final both absent): {n_missing_program}")
print(f"Reflection changed the program: {n_reflection_changed} ({100 * n_reflection_changed / len(results):.1f}%)")
print(f"Traces with leak phrases: {n_leak}")


## Save all results for review

Saves every sample's raw output + parsed fields + validation flags, regardless
of pass/fail, so the reflection behavior and exact-match rate can be reviewed
before deciding on SFT filtering (keep `exact_program_match == True` only, same
as Chi's pipeline).

In [ ]:
output_path = OUTPUT_DIR / "conr_trace_v2_independent_solve.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "n_samples": len(results),
        "summary": {
            "exact_program_match": n_exact_match,
            "missing_program": n_missing_program,
            "reflection_changed_program": n_reflection_changed,
            "leak_phrases": n_leak,
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} results to {output_path}")
print("Download this file from the Server Web UI file browser.")
